# 0. IMPORT THƯ VIỆN

In [14]:
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
import matplotlib.cm as cm
from datetime import timedelta
import os

# 1. CẤU HÌNH ĐỒ HOẠ

In [15]:
PALETTE_PRIMARY  = ["#1a1a2e", "#16213e", "#0f3460", "#e94560", "#f5a623", "#50fa7b", "#8be9fd"]
PALETTE_SEGMENT  = ["#e94560", "#f5a623", "#50fa7b", "#8be9fd", "#bd93f9", "#ff79c6"]
PALETTE_HEAT     = "YlOrRd"
BG_COLOR         = "#0d1117"
TEXT_COLOR       = "#e6edf3"
GRID_COLOR       = "#21262d"
ACCENT           = "#e94560"
ACCENT2          = "#f5a623"
ACCENT3          = "#50fa7b"
 
def set_style():
    plt.rcParams.update({
        "figure.facecolor":  BG_COLOR,
        "axes.facecolor":    "#161b22",
        "axes.edgecolor":    GRID_COLOR,
        "axes.labelcolor":   TEXT_COLOR,
        "axes.titlecolor":   TEXT_COLOR,
        "axes.titlesize":    13,
        "axes.labelsize":    11,
        "xtick.color":       TEXT_COLOR,
        "ytick.color":       TEXT_COLOR,
        "xtick.labelsize":   9,
        "ytick.labelsize":   9,
        "legend.facecolor":  "#161b22",
        "legend.edgecolor":  GRID_COLOR,
        "legend.labelcolor": TEXT_COLOR,
        "text.color":        TEXT_COLOR,
        "grid.color":        GRID_COLOR,
        "grid.linewidth":    0.5,
        "grid.alpha":        0.6,
        "font.family":       "monospace",
        "figure.dpi":        130,
    })
 
set_style()
 
OUTPUT_DIR = "eda_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
def save_fig(name):
    plt.savefig(f"{OUTPUT_DIR}/{name}.png", bbox_inches="tight",
                facecolor=BG_COLOR, dpi=130)
    plt.close()
    print(f"  ✓ Saved: {OUTPUT_DIR}/{name}.png")

# 2. NẠP DỮ LIỆU

In [19]:
print("=" * 60)
print("  DATATHON 2026 — Customer Analytics EDA")
print("=" * 60)

DATA_DIR = "."   # Thay đổi nếu data ở thư mục khác

def load(filename, parse_dates=None):
    path = os.path.join(DATA_DIR, filename)
    return pd.read_csv(path, parse_dates=parse_dates)

print("\n[+] Loading data...")
customers   = load("Data\customers.csv",   parse_dates=["signup_date"])
orders      = load("Data\orders.csv",      parse_dates=["order_date"])
order_items = load("Data\order_items.csv")
payments    = load("Data\payments.csv")
products    = load("Data\products.csv")
returns   = load(r"Data\returns.csv",   parse_dates=["return_date"])
reviews     = load(r"Data\reviews.csv",     parse_dates=["review_date"])
shipments   = load("Data\shipments.csv",   parse_dates=["ship_date","delivery_date"])
geography   = load("Data\geography.csv")
sales_train = load("Data\sales.csv",       parse_dates=["Date"])

print(f"  customers   : {customers.shape}")
print(f"  orders      : {orders.shape}")
print(f"  order_items : {order_items.shape}")
print(f"  payments    : {payments.shape}")
print(f"  products    : {products.shape}")
print(f"  returns     : {returns.shape}")
print(f"  reviews     : {reviews.shape}")
print(f"  shipments   : {shipments.shape}")
print(f"  geography   : {geography.shape}")
print(f"  sales_train : {sales_train.shape}")


  DATATHON 2026 — Customer Analytics EDA

[+] Loading data...
  customers   : (121930, 7)
  orders      : (646945, 8)
  order_items : (714669, 7)
  payments    : (646945, 4)
  products    : (2412, 8)
  returns     : (39939, 7)
  reviews     : (113551, 7)
  shipments   : (566067, 4)
  geography   : (39948, 4)
  sales_train : (3833, 3)


# 3. FEATURE ENGINEERING

In [20]:
print("\n[+] Feature engineering...")

# ---- Merge orders + payments + geography ----
orders_full = (
    orders
    .merge(payments[["order_id","payment_value"]], on="order_id", how="left")
    .merge(geography[["zip","region","city"]], on="zip",  how="left")
)



[+] Feature engineering...


In [21]:
# ---- Customer-level aggregations ----
cust_orders = (
    orders_full
    .groupby("customer_id")
    .agg(
        n_orders       = ("order_id",       "nunique"),
        total_spend    = ("payment_value",  "sum"),
        first_order    = ("order_date",     "min"),
        last_order     = ("order_date",     "max"),
        cancelled_cnt  = ("order_status",   lambda x: (x=="cancelled").sum()),
        delivered_cnt  = ("order_status",   lambda x: (x=="delivered").sum()),
    )
    .reset_index()
)
cust_orders["avg_order_value"] = cust_orders["total_spend"] / cust_orders["n_orders"]
cust_orders["lifespan_days"]   = (cust_orders["last_order"] - cust_orders["first_order"]).dt.days
cust_orders["cancel_rate"]     = cust_orders["cancelled_cnt"] / cust_orders["n_orders"]


In [22]:
# ---- Inter-order gap (Q1 answer) ----
orders_sorted = orders_full.sort_values(["customer_id","order_date"])
orders_sorted["prev_order"] = orders_sorted.groupby("customer_id")["order_date"].shift(1)
orders_sorted["gap_days"]   = (orders_sorted["order_date"] - orders_sorted["prev_order"]).dt.days
gap_median = orders_sorted.dropna(subset=["gap_days"])["gap_days"].median()
print(f"  Inter-order gap median: {gap_median:.1f} days")

# ---- Merge với customers ----
cust_full = customers.merge(cust_orders, on="customer_id", how="left")
cust_full["has_order"] = cust_full["n_orders"].notna()
cust_full["n_orders"]  = cust_full["n_orders"].fillna(0)

# ---- Tenure ----
REF_DATE = pd.Timestamp("2023-01-01")
cust_full["tenure_days"] = (REF_DATE - cust_full["signup_date"]).dt.days

# ---- RFM ----
rfm = cust_orders.copy()
rfm["recency"]   = (REF_DATE - rfm["last_order"]).dt.days
rfm["frequency"] = rfm["n_orders"]
rfm["monetary"]  = rfm["total_spend"]

# Score 1-4 (quartile)
for col, label in [("recency","R"),("frequency","F"),("monetary","M")]:
    try:
        if col == "recency":
            rfm[label] = pd.qcut(rfm[col], 4, labels=[4,3,2,1]).astype(int)
        else:
            rfm[label] = pd.qcut(rfm[col].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
    except Exception:
        rfm[label] = 2

rfm["RFM_Score"] = rfm["R"].astype(str) + rfm["F"].astype(str) + rfm["M"].astype(str)
rfm["RFM_Total"] = rfm["R"] + rfm["F"] + rfm["M"]

def segment_rfm(row):
    r, f, m = row["R"], row["F"], row["M"]
    if r >= 4 and f >= 4 and m >= 4:  return "Champions"
    if r >= 3 and f >= 3:             return "Loyal"
    if r >= 4 and f <= 1:             return "New Customer"
    if r >= 3 and f <= 2:             return "Promising"
    if r <= 2 and f >= 3:             return "At Risk"
    if r == 1 and f >= 2:             return "Lost"
    return "Others"

rfm["Segment"] = rfm.apply(segment_rfm, axis=1)
rfm = rfm.merge(customers[["customer_id","gender","age_group",
                            "acquisition_channel","city"]], on="customer_id", how="left")

print(f"  RFM segments: {rfm['Segment'].value_counts().to_dict()}")


  Inter-order gap median: 144.0 days
  RFM segments: {'Others': 27264, 'Loyal': 22791, 'Champions': 11521, 'At Risk': 10811, 'Promising': 9688, 'Lost': 7033, 'New Customer': 1138}


In [23]:
# ---- Return rate per customer ----
returns_enriched = returns.merge(order_items[["order_id","product_id","quantity"]],
                                  on=["order_id","product_id"], how="left")
returns_enriched = returns_enriched.merge(orders[["order_id","customer_id"]], on="order_id", how="left")
cust_return = (returns_enriched.groupby("customer_id")["return_quantity"].sum()
               .reset_index().rename(columns={"return_quantity":"total_returned"}))
rfm = rfm.merge(cust_return, on="customer_id", how="left")
rfm["total_returned"] = rfm["total_returned"].fillna(0)

# ---- Review stats per customer ----
cust_reviews = (reviews.groupby("customer_id")["rating"]
                .agg(avg_rating="mean", n_reviews="count").reset_index())
rfm = rfm.merge(cust_reviews, on="customer_id", how="left")


In [24]:
# ══════════════════════════════════════════════
# ████  DESCRIPTIVE  ████
# ══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  [DESCRIPTIVE] — What happened?")
print("=" * 60)

# ────────────────────────────────────────────
# FIG 1: Customer Overview Dashboard
# ────────────────────────────────────────────
print("[Fig 1] Customer Overview Dashboard...")

fig = plt.figure(figsize=(20, 12))
fig.suptitle("CUSTOMER OVERVIEW DASHBOARD", fontsize=18, fontweight="bold",
             color=ACCENT, y=0.98)
gs = GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.42)

# KPI boxes
kpis = [
    ("Total Customers",  f"{len(customers):,}",         ACCENT),
    ("Active Customers", f"{(cust_full['n_orders']>0).sum():,}", ACCENT2),
    ("Avg Orders / Cust",f"{cust_full['n_orders'].mean():.1f}", ACCENT3),
    ("Avg Spend / Cust", f"${cust_full['total_spend'].mean():,.0f}", "#bd93f9"),
]
for i, (label, val, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor("#0d1117")
    ax.text(0.5, 0.65, val,    ha="center", va="center", fontsize=22,
            fontweight="bold", color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label,  ha="center", va="center", fontsize=9,
            color=TEXT_COLOR, transform=ax.transAxes)
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(1.5)
    ax.set_xticks([]); ax.set_yticks([])

# Gender distribution
ax1 = fig.add_subplot(gs[1, 0])
gender_c = customers["gender"].value_counts(dropna=False)
gender_c.index = [str(x) if pd.notna(x) else "Unknown" for x in gender_c.index]
bars = ax1.bar(gender_c.index, gender_c.values,
               color=[ACCENT, ACCENT2, ACCENT3][:len(gender_c)], edgecolor="none", width=0.6)
ax1.set_title("Gender Distribution", pad=8)
ax1.set_ylabel("Customers")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax1.grid(axis="y", alpha=0.4)
for bar in bars:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
             f"{bar.get_height():,}", ha="center", fontsize=8, color=TEXT_COLOR)

# Age group distribution
ax2 = fig.add_subplot(gs[1, 1])
age_order = ["18-24","25-34","35-44","45-54","55+"]
age_c = customers["age_group"].value_counts().reindex(age_order, fill_value=0)
bars2 = ax2.bar(age_c.index, age_c.values, color=PALETTE_SEGMENT[:len(age_c)], edgecolor="none", width=0.7)
ax2.set_title("Age Group Distribution", pad=8)
ax2.set_ylabel("Customers")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax2.grid(axis="y", alpha=0.4)
for bar in bars2:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
             f"{bar.get_height():,}", ha="center", fontsize=8, color=TEXT_COLOR)

# Acquisition channel
ax3 = fig.add_subplot(gs[1, 2])
acq_c = customers["acquisition_channel"].value_counts(dropna=False)
acq_c.index = [str(x) if pd.notna(x) else "Unknown" for x in acq_c.index]
wedges, texts, autotexts = ax3.pie(
    acq_c.values, labels=None,
    autopct="%1.1f%%", pctdistance=0.75,
    colors=PALETTE_SEGMENT[:len(acq_c)],
    wedgeprops=dict(edgecolor=BG_COLOR, linewidth=1.5),
    startangle=90
)
for at in autotexts: at.set_fontsize(7); at.set_color(TEXT_COLOR)
ax3.set_title("Acquisition Channel", pad=8)
ax3.legend(acq_c.index, loc="lower center", bbox_to_anchor=(0.5,-0.25),
           ncol=2, fontsize=7, framealpha=0)

# Monthly signups
ax4 = fig.add_subplot(gs[1, 3])
customers["signup_ym"] = customers["signup_date"].dt.to_period("M")
signup_monthly = customers.groupby("signup_ym").size()
ax4.fill_between(range(len(signup_monthly)), signup_monthly.values,
                 alpha=0.4, color=ACCENT2)
ax4.plot(range(len(signup_monthly)), signup_monthly.values,
         color=ACCENT2, linewidth=1.5)
ax4.set_title("Monthly New Signups", pad=8)
ax4.set_ylabel("New Customers")
ax4.set_xticks([])
ax4.grid(axis="y", alpha=0.4)
ax4.text(0.02, 0.92, f"Peak: {signup_monthly.max():,}",
         transform=ax4.transAxes, fontsize=8, color=ACCENT2)

# Order frequency distribution
ax5 = fig.add_subplot(gs[2, 0:2])
order_freq = cust_full["n_orders"].clip(upper=20).value_counts().sort_index()
ax5.bar(order_freq.index, order_freq.values,
        color=ACCENT, edgecolor="none", width=0.85)
ax5.set_title("Order Frequency Distribution (orders per customer, capped @20)", pad=8)
ax5.set_xlabel("Number of Orders")
ax5.set_ylabel("Customer Count")
ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax5.grid(axis="y", alpha=0.4)

# Spend distribution
ax6 = fig.add_subplot(gs[2, 2:4])
spends = cust_orders["total_spend"].dropna()
ax6.hist(spends.clip(upper=spends.quantile(0.99)), bins=50,
         color=ACCENT3, edgecolor="none", alpha=0.85)
ax6.set_title("Total Spend Distribution (99th pct clipped)", pad=8)
ax6.set_xlabel("Total Spend ($)")
ax6.set_ylabel("Customer Count")
ax6.axvline(spends.median(), color=ACCENT, linestyle="--", linewidth=1.5,
            label=f"Median: ${spends.median():,.0f}")
ax6.axvline(spends.mean(), color=ACCENT2, linestyle="--", linewidth=1.5,
            label=f"Mean: ${spends.mean():,.0f}")
ax6.legend(fontsize=8)
ax6.grid(axis="y", alpha=0.4)

save_fig("01_customer_overview")



  [DESCRIPTIVE] — What happened?
[Fig 1] Customer Overview Dashboard...
  ✓ Saved: eda_outputs/01_customer_overview.png


In [25]:
# ────────────────────────────────────────────
# FIG 2: Customer Cohort — Monthly Signup & City Heatmap
# ────────────────────────────────────────────
print("[Fig 2] Signup trends & geography...")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("CUSTOMER GEOGRAPHY & SIGNUP COHORT", fontsize=15,
             fontweight="bold", color=ACCENT)

# City top 15
ax = axes[0]
city_c = customers.merge(geography[["zip","region"]], on="zip", how="left")
city_top = city_c["city_x"].value_counts().head(15) if "city_x" in city_c.columns \
           else city_c["city"].value_counts().head(15)
# handle column name after merge
city_col = "city_x" if "city_x" in city_c.columns else "city"
city_top = city_c[city_col].value_counts().head(15)
colors_bar = [ACCENT if i < 3 else ACCENT2 if i < 7 else "#555" for i in range(15)]
ax.barh(city_top.index[::-1], city_top.values[::-1], color=colors_bar[::-1], edgecolor="none")
ax.set_title("Top 15 Cities by Customer Count", pad=8)
ax.set_xlabel("Customers")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax.grid(axis="x", alpha=0.4)
for i, (idx, val) in enumerate(zip(city_top.index[::-1], city_top.values[::-1])):
    ax.text(val + 20, i, f"{val:,}", va="center", fontsize=7.5, color=TEXT_COLOR)

# Year-month heatmap of signups
ax2 = axes[1]
customers["signup_year"]  = customers["signup_date"].dt.year
customers["signup_month"] = customers["signup_date"].dt.month
pivot = customers.pivot_table(index="signup_year", columns="signup_month",
                               aggfunc="size", fill_value=0)
pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
sns.heatmap(pivot, ax=ax2, cmap="YlOrRd", annot=True, fmt="d",
            linewidths=0.5, linecolor=BG_COLOR,
            annot_kws={"fontsize":7}, cbar_kws={"shrink":0.7})
ax2.set_title("Monthly Signup Heatmap (Year × Month)", pad=8)
ax2.set_xlabel("Month"); ax2.set_ylabel("Year")
ax2.tick_params(colors=TEXT_COLOR)

plt.tight_layout(rect=[0,0,1,0.94])
save_fig("02_customer_geo_signup")

[Fig 2] Signup trends & geography...
  ✓ Saved: eda_outputs/02_customer_geo_signup.png


In [26]:
# ══════════════════════════════════════════════
# ████  DIAGNOSTIC  ████
# ══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  [DIAGNOSTIC] — Why did it happen?")
print("=" * 60)

# ────────────────────────────────────────────
# FIG 3: RFM Segmentation
# ────────────────────────────────────────────
print("[Fig 3] RFM Segmentation...")

seg_order = ["Champions","Loyal","Promising","New Customer","At Risk","Lost","Others"]
seg_colors = {
    "Champions":    ACCENT3,
    "Loyal":        "#8be9fd",
    "Promising":    ACCENT2,
    "New Customer": "#bd93f9",
    "At Risk":      ACCENT,
    "Lost":         "#ff5555",
    "Others":       "#555555",
}

seg_stats = (rfm.groupby("Segment")
             .agg(
                 count     = ("customer_id","count"),
                 avg_spend = ("monetary",   "mean"),
                 avg_freq  = ("frequency",  "mean"),
                 avg_rec   = ("recency",    "mean"),
             )
             .reindex(seg_order)
             .fillna(0)
             .reset_index())

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
fig.suptitle("RFM CUSTOMER SEGMENTATION ANALYSIS", fontsize=16,
             fontweight="bold", color=ACCENT)

# Pie of segment size
ax = axes[0][0]
seg_colors_list = [seg_colors.get(s, "#555") for s in seg_stats["Segment"]]
wedges, _, autotexts = ax.pie(
    seg_stats["count"], labels=None,
    autopct=lambda p: f"{p:.1f}%\n({int(p*rfm.shape[0]/100):,})" if p > 3 else "",
    colors=seg_colors_list, startangle=140,
    wedgeprops=dict(edgecolor=BG_COLOR, linewidth=2), pctdistance=0.75
)
for at in autotexts: at.set_fontsize(7); at.set_color(TEXT_COLOR)
ax.set_title("Customer Segment Distribution", pad=8)
ax.legend(seg_stats["Segment"], loc="lower left", fontsize=7.5, framealpha=0)

# Avg spend per segment
ax2 = axes[0][1]
bars = ax2.bar(seg_stats["Segment"], seg_stats["avg_spend"],
               color=seg_colors_list, edgecolor="none", width=0.7)
ax2.set_title("Average Total Spend by Segment", pad=8)
ax2.set_ylabel("Avg Spend ($)")
ax2.set_xticklabels(seg_stats["Segment"], rotation=25, ha="right", fontsize=8)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax2.grid(axis="y", alpha=0.4)
for bar, val in zip(bars, seg_stats["avg_spend"]):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
             f"${val:,.0f}", ha="center", fontsize=7.5, color=TEXT_COLOR)

# Scatter Recency vs Frequency (sample)
ax3 = axes[1][0]
sample = rfm.sample(min(3000, len(rfm)), random_state=42)
for seg, grp in sample.groupby("Segment"):
    ax3.scatter(grp["recency"], grp["frequency"],
                c=seg_colors.get(seg,"#555"), label=seg,
                alpha=0.5, s=18, edgecolors="none")
ax3.set_xlabel("Recency (days since last order)")
ax3.set_ylabel("Frequency (orders)")
ax3.set_title("Recency vs Frequency by Segment", pad=8)
ax3.legend(fontsize=7.5, markerscale=1.5)
ax3.grid(alpha=0.3)

# Heatmap: Segment × Age Group (% share)
ax4 = axes[1][1]
seg_age = (rfm.dropna(subset=["age_group"])
             .groupby(["Segment","age_group"]).size()
             .unstack(fill_value=0))
seg_age_pct = seg_age.div(seg_age.sum(axis=1), axis=0) * 100
seg_age_pct = seg_age_pct.reindex(seg_order).fillna(0)
age_cols = ["18-24","25-34","35-44","45-54","55+"]
seg_age_pct = seg_age_pct.reindex(columns=age_cols, fill_value=0)
sns.heatmap(seg_age_pct, ax=ax4, cmap="YlOrRd", annot=True, fmt=".1f",
            linewidths=0.5, linecolor=BG_COLOR,
            annot_kws={"fontsize":8}, cbar_kws={"shrink":0.7, "label":"% within segment"})
ax4.set_title("Segment × Age Group (% distribution)", pad=8)
ax4.set_xlabel("Age Group"); ax4.set_ylabel("Segment")

plt.tight_layout(rect=[0,0,1,0.95])
save_fig("03_rfm_segmentation")



  [DIAGNOSTIC] — Why did it happen?
[Fig 3] RFM Segmentation...
  ✓ Saved: eda_outputs/03_rfm_segmentation.png


In [27]:
# ────────────────────────────────────────────
# FIG 4: Churn Diagnostics — Cancel & Return Behaviour
# ────────────────────────────────────────────
print("[Fig 4] Churn diagnostics...")

# Cancel rate by age group
cancel_by_age = (
    orders_full.merge(customers[["customer_id","age_group","gender"]], on="customer_id", how="left")
    .assign(cancelled = lambda df: (df["order_status"] == "cancelled").astype(int))
    .groupby("age_group")
    .agg(total=("order_id","count"), cancelled=("cancelled","sum"))
    .assign(cancel_rate = lambda df: df["cancelled"]/df["total"]*100)
    .reindex(["18-24","25-34","35-44","45-54","55+"])
    .dropna()
)

# Cancel rate by acquisition channel
cancel_by_acq = (
    orders_full.merge(customers[["customer_id","acquisition_channel"]], on="customer_id", how="left")
    .assign(cancelled = lambda df: (df["order_status"] == "cancelled").astype(int))
    .groupby("acquisition_channel")
    .agg(total=("order_id","count"), cancelled=("cancelled","sum"))
    .assign(cancel_rate = lambda df: df["cancelled"]/df["total"]*100)
    .dropna()
    .sort_values("cancel_rate", ascending=False)
)

# Return reason distribution
return_reason = returns["return_reason"].value_counts()

# Return rate by segment
ret_by_seg = (
    rfm[["customer_id","Segment","n_orders","total_returned"]].copy()
    .assign(ret_rate_est = lambda df: df["total_returned"] / (df["n_orders"]+1) * 100)
    .groupby("Segment")["ret_rate_est"].mean()
    .reindex(seg_order).fillna(0)
)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle("CHURN & RETURN DIAGNOSTICS", fontsize=15, fontweight="bold", color=ACCENT)

ax = axes[0][0]
colors_age = [ACCENT if v == cancel_by_age["cancel_rate"].max() else ACCENT2
              for v in cancel_by_age["cancel_rate"]]
bars = ax.bar(cancel_by_age.index, cancel_by_age["cancel_rate"],
              color=colors_age, edgecolor="none", width=0.65)
ax.set_title("Cancellation Rate by Age Group", pad=8)
ax.set_ylabel("Cancel Rate (%)")
ax.grid(axis="y", alpha=0.4)
for bar, val in zip(bars, cancel_by_age["cancel_rate"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f"{val:.1f}%", ha="center", fontsize=8.5, color=TEXT_COLOR)

ax2 = axes[0][1]
colors_acq = [ACCENT if i == 0 else ACCENT2 if i == 1 else "#555"
              for i in range(len(cancel_by_acq))]
ax2.barh(cancel_by_acq.index[::-1], cancel_by_acq["cancel_rate"][::-1],
         color=colors_acq[::-1], edgecolor="none")
ax2.set_title("Cancellation Rate by Acquisition Channel", pad=8)
ax2.set_xlabel("Cancel Rate (%)")
ax2.grid(axis="x", alpha=0.4)
for i, val in enumerate(cancel_by_acq["cancel_rate"][::-1]):
    ax2.text(val + 0.05, i, f"{val:.1f}%", va="center", fontsize=8, color=TEXT_COLOR)

ax3 = axes[1][0]
bars3 = ax3.bar(return_reason.index, return_reason.values,
                color=PALETTE_SEGMENT[:len(return_reason)], edgecolor="none", width=0.7)
ax3.set_title("Return Reason Distribution (all categories)", pad=8)
ax3.set_ylabel("Return Records")
ax3.set_xticklabels(return_reason.index, rotation=30, ha="right", fontsize=8)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax3.grid(axis="y", alpha=0.4)
for bar, val in zip(bars3, return_reason.values):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
             f"{val:,}", ha="center", fontsize=7.5, color=TEXT_COLOR)

ax4 = axes[1][1]
colors_seg = [seg_colors.get(s,"#555") for s in ret_by_seg.index]
bars4 = ax4.bar(ret_by_seg.index, ret_by_seg.values, color=colors_seg, edgecolor="none", width=0.7)
ax4.set_title("Estimated Return Rate by Customer Segment", pad=8)
ax4.set_ylabel("Est. Return Rate (%)")
ax4.set_xticklabels(ret_by_seg.index, rotation=25, ha="right", fontsize=8)
ax4.grid(axis="y", alpha=0.4)
for bar, val in zip(bars4, ret_by_seg.values):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f"{val:.1f}%", ha="center", fontsize=8, color=TEXT_COLOR)

plt.tight_layout(rect=[0,0,1,0.94])
save_fig("04_churn_return_diagnostics")


[Fig 4] Churn diagnostics...
  ✓ Saved: eda_outputs/04_churn_return_diagnostics.png


In [28]:
# ────────────────────────────────────────────
# FIG 5: Inter-order Gap & Repeat Purchase Behaviour
# ────────────────────────────────────────────
print("[Fig 5] Inter-order gap analysis...")

gaps = orders_sorted.dropna(subset=["gap_days"])

# By age group
gaps_age = gaps.merge(customers[["customer_id","age_group"]], on="customer_id", how="left")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("REPEAT PURCHASE & INTER-ORDER GAP ANALYSIS", fontsize=15,
             fontweight="bold", color=ACCENT)

# Overall gap distribution
ax = axes[0]
ax.hist(gaps["gap_days"].clip(upper=400), bins=60, color=ACCENT, edgecolor="none", alpha=0.85)
ax.axvline(gap_median, color=ACCENT2, linewidth=2, linestyle="--",
           label=f"Median: {gap_median:.0f} days")
ax.axvline(gaps["gap_days"].mean(), color=ACCENT3, linewidth=2, linestyle=":",
           label=f"Mean: {gaps['gap_days'].mean():.0f} days")
ax.set_title("Inter-Order Gap Distribution (days)", pad=8)
ax.set_xlabel("Days Between Orders")
ax.set_ylabel("Count")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.4)

# Gap by age group — box
ax2 = axes[1]
age_groups = ["18-24","25-34","35-44","45-54","55+"]
data_by_age = [gaps_age[gaps_age["age_group"]==ag]["gap_days"].clip(upper=400).dropna().values
               for ag in age_groups]
bp = ax2.boxplot(data_by_age, labels=age_groups, patch_artist=True,
                 medianprops=dict(color=ACCENT, linewidth=2),
                 whiskerprops=dict(color=TEXT_COLOR),
                 capprops=dict(color=TEXT_COLOR),
                 flierprops=dict(marker=".", color=TEXT_COLOR, alpha=0.3, markersize=3))
for patch, color in zip(bp["boxes"], PALETTE_SEGMENT):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax2.set_title("Inter-Order Gap by Age Group", pad=8)
ax2.set_xlabel("Age Group"); ax2.set_ylabel("Gap (days)")
ax2.grid(axis="y", alpha=0.4)

# Repeat purchase rate
ax3 = axes[2]
repeat_stats = (
    orders.groupby("customer_id")["order_id"].count()
    .reset_index()
    .assign(is_repeat = lambda df: df["order_id"] > 1)
    .merge(customers[["customer_id","acquisition_channel"]], on="customer_id", how="left")
    .groupby("acquisition_channel")["is_repeat"]
    .mean()
    .sort_values(ascending=False)
    .dropna()
    * 100
)
colors_rep = [ACCENT3 if v == repeat_stats.max() else ACCENT2 if v > repeat_stats.median()
              else ACCENT for v in repeat_stats.values]
ax3.barh(repeat_stats.index[::-1], repeat_stats.values[::-1],
         color=colors_rep[::-1], edgecolor="none")
ax3.set_title("Repeat Purchase Rate by Channel", pad=8)
ax3.set_xlabel("Repeat Rate (%)")
ax3.grid(axis="x", alpha=0.4)
for i, val in enumerate(repeat_stats.values[::-1]):
    ax3.text(val+0.3, i, f"{val:.1f}%", va="center", fontsize=8, color=TEXT_COLOR)

plt.tight_layout(rect=[0,0,1,0.94])
save_fig("05_repeat_purchase_gap")


[Fig 5] Inter-order gap analysis...
  ✓ Saved: eda_outputs/05_repeat_purchase_gap.png


In [29]:
# ────────────────────────────────────────────
# FIG 6: Review & Satisfaction Analysis
# ────────────────────────────────────────────
print("[Fig 6] Review & satisfaction analysis...")

# Rating distribution overall
rating_dist = reviews["rating"].value_counts().sort_index()

# Avg rating by segment
rat_seg = (rfm.dropna(subset=["avg_rating"])
             .groupby("Segment")["avg_rating"].mean()
             .reindex(seg_order).fillna(0))

# Rating by product category
rat_cat = (reviews.merge(products[["product_id","category"]], on="product_id", how="left")
             .groupby("category")["rating"].agg(["mean","count"])
             .sort_values("mean", ascending=False))

# Monthly avg rating trend
reviews["review_ym"] = reviews["review_date"].dt.to_period("M")
rat_monthly = reviews.groupby("review_ym")["rating"].mean()

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle("CUSTOMER SATISFACTION & REVIEW ANALYSIS", fontsize=15,
             fontweight="bold", color=ACCENT)

ax = axes[0][0]
bar_colors = [ACCENT if i < 2 else ACCENT2 if i == 3 else ACCENT3 for i in range(5)]
bars = ax.bar(rating_dist.index, rating_dist.values, color=bar_colors,
              edgecolor="none", width=0.7)
ax.set_title("Overall Rating Distribution (1–5 Stars)", pad=8)
ax.set_xlabel("Rating"); ax.set_ylabel("Review Count")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax.grid(axis="y", alpha=0.4)
for bar, val in zip(bars, rating_dist.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            f"{val:,}\n({val/rating_dist.sum()*100:.1f}%)",
            ha="center", fontsize=7.5, color=TEXT_COLOR)
ax.text(0.98, 0.95, f"Avg: {reviews['rating'].mean():.2f} ★",
        transform=ax.transAxes, ha="right", fontsize=10, color=ACCENT2, fontweight="bold")

ax2 = axes[0][1]
colors_seg2 = [seg_colors.get(s,"#555") for s in rat_seg.index]
bars2 = ax2.bar(rat_seg.index, rat_seg.values, color=colors_seg2, edgecolor="none", width=0.7)
ax2.set_title("Average Rating by Customer Segment", pad=8)
ax2.set_ylabel("Avg Rating")
ax2.set_ylim(0, 5.5)
ax2.set_xticklabels(rat_seg.index, rotation=25, ha="right", fontsize=8)
ax2.axhline(reviews["rating"].mean(), color=ACCENT, linestyle="--", linewidth=1.5,
            label=f"Overall avg: {reviews['rating'].mean():.2f}")
ax2.legend(fontsize=8)
ax2.grid(axis="y", alpha=0.4)
for bar, val in zip(bars2, rat_seg.values):
    if val > 0:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f"{val:.2f}", ha="center", fontsize=8, color=TEXT_COLOR)

ax3 = axes[1][0]
bars3 = ax3.barh(rat_cat.index[::-1], rat_cat["mean"][::-1],
                 color=PALETTE_SEGMENT[:len(rat_cat)], edgecolor="none")
ax3.set_title("Average Rating by Product Category", pad=8)
ax3.set_xlabel("Avg Rating")
ax3.set_xlim(0, 5.5)
ax3.grid(axis="x", alpha=0.4)
for i, (val, cnt) in enumerate(zip(rat_cat["mean"][::-1], rat_cat["count"][::-1])):
    ax3.text(val+0.05, i, f"{val:.2f} (n={cnt:,})", va="center", fontsize=8, color=TEXT_COLOR)

ax4 = axes[1][1]
ax4.plot(range(len(rat_monthly)), rat_monthly.values, color=ACCENT2, linewidth=1.5)
ax4.fill_between(range(len(rat_monthly)), rat_monthly.values,
                 alpha=0.25, color=ACCENT2)
ax4.axhline(rat_monthly.mean(), color=ACCENT, linestyle="--", linewidth=1,
            label=f"Mean: {rat_monthly.mean():.2f}")
ax4.set_title("Monthly Average Rating Trend", pad=8)
ax4.set_ylabel("Avg Rating"); ax4.set_xticks([])
ax4.set_ylim(1, 5)
ax4.legend(fontsize=8)
ax4.grid(axis="y", alpha=0.4)

plt.tight_layout(rect=[0,0,1,0.94])
save_fig("06_review_satisfaction")


[Fig 6] Review & satisfaction analysis...
  ✓ Saved: eda_outputs/06_review_satisfaction.png


In [30]:
# ══════════════════════════════════════════════
# ████  PREDICTIVE  ████
# ══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  [PREDICTIVE] — What is likely to happen?")
print("=" * 60)

# ────────────────────────────────────────────
# FIG 7: Customer LTV Prediction via KMeans Clustering
# ────────────────────────────────────────────
print("[Fig 7] KMeans clustering for LTV prediction...")

# Prepare features for clustering
cluster_df = rfm[["customer_id","recency","frequency","monetary"]].dropna()
features = ["recency","frequency","monetary"]
X = cluster_df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow + silhouette
inertias, silhouettes = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

best_k = K_range[np.argmax(silhouettes)]
print(f"  Best K (max silhouette): {best_k}")

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_df = cluster_df.copy()
cluster_df["cluster"] = km_final.fit_predict(X_scaled)

# Cluster summary
cluster_summary = (
    cluster_df.groupby("cluster")[features]
    .mean()
    .assign(count=cluster_df.groupby("cluster").size())
    .sort_values("monetary", ascending=False)
    .reset_index()
)
cluster_summary["label"] = [f"C{i}: " + (
    "High-Value" if i==0 else "Mid-Value" if i==1 else
    "Low-Value" if i==2 else "Dormant" if i==3 else f"Cluster {i}")
    for i in range(len(cluster_summary))]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("CUSTOMER CLUSTERING — LTV PREDICTION", fontsize=15,
             fontweight="bold", color=ACCENT)

ax = axes[0]
ax.plot(list(K_range), silhouettes, marker="o", color=ACCENT2, linewidth=2)
ax.axvline(best_k, color=ACCENT, linestyle="--", linewidth=1.5,
           label=f"Best K={best_k}")
ax.set_title("Silhouette Score vs K", pad=8)
ax.set_xlabel("K (Number of Clusters)")
ax.set_ylabel("Silhouette Score")
ax.legend(fontsize=9); ax.grid(alpha=0.4)

ax2 = axes[1]
cluster_colors = PALETTE_SEGMENT[:best_k]
sample_cl = cluster_df.sample(min(2000, len(cluster_df)), random_state=42)
for cid in range(best_k):
    mask = sample_cl["cluster"] == cid
    ax2.scatter(sample_cl.loc[mask,"recency"], sample_cl.loc[mask,"monetary"],
                c=cluster_colors[cid], label=cluster_summary.loc[cid,"label"],
                alpha=0.5, s=15, edgecolors="none")
ax2.set_xlabel("Recency (days)"); ax2.set_ylabel("Monetary (total spend)")
ax2.set_title("Cluster: Recency vs Monetary", pad=8)
ax2.legend(fontsize=7.5, markerscale=2)
ax2.grid(alpha=0.3)

ax3 = axes[2]
x_pos = np.arange(len(cluster_summary))
bars = ax3.bar(x_pos, cluster_summary["monetary"],
               color=cluster_colors, edgecolor="none", width=0.65)
ax3.set_title("Avg Monetary Value per Cluster", pad=8)
ax3.set_ylabel("Avg Total Spend ($)")
ax3.set_xticks(x_pos)
ax3.set_xticklabels(cluster_summary["label"], rotation=20, ha="right", fontsize=7.5)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax3.grid(axis="y", alpha=0.4)
for bar, row in zip(bars, cluster_summary.itertuples()):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
             f"n={row.count:,}", ha="center", fontsize=7.5, color=TEXT_COLOR)

plt.tight_layout(rect=[0,0,1,0.94])
save_fig("07_ltv_clustering")



  [PREDICTIVE] — What is likely to happen?
[Fig 7] KMeans clustering for LTV prediction...
  Best K (max silhouette): 2
  ✓ Saved: eda_outputs/07_ltv_clustering.png


In [31]:
# ────────────────────────────────────────────
# FIG 8: Cohort Retention Analysis
# ────────────────────────────────────────────
print("[Fig 8] Cohort retention analysis...")

orders_cust = orders.merge(customers[["customer_id","signup_date"]], on="customer_id", how="left")
orders_cust["cohort_month"] = orders_cust["signup_date"].dt.to_period("M")
orders_cust["order_month"]  = orders_cust["order_date"].dt.to_period("M")
orders_cust["period_num"]   = (orders_cust["order_month"] - orders_cust["cohort_month"]).apply(
    lambda x: x.n if hasattr(x,"n") else 0)

cohort_data = (orders_cust.groupby(["cohort_month","period_num"])["customer_id"]
               .nunique().reset_index())
cohort_sizes = cohort_data[cohort_data["period_num"]==0].set_index("cohort_month")["customer_id"]
cohort_pivot = cohort_data.pivot(index="cohort_month", columns="period_num",
                                  values="customer_id")
cohort_pct = cohort_pivot.div(cohort_sizes, axis=0) * 100

# Keep first 18 periods and most recent 24 cohorts for readability
cohort_pct_plot = cohort_pct.iloc[-24:, :18]

fig, ax = plt.subplots(figsize=(20, 9))
fig.suptitle("CUSTOMER COHORT RETENTION HEATMAP", fontsize=15,
             fontweight="bold", color=ACCENT)

sns.heatmap(cohort_pct_plot, ax=ax, cmap="YlOrRd_r", annot=True,
            fmt=".0f", linewidths=0.4, linecolor=BG_COLOR,
            annot_kws={"fontsize":6.5}, vmin=0, vmax=100,
            cbar_kws={"shrink":0.5, "label":"Retention %"})

ax.set_xlabel("Period (months since first order)", labelpad=10)
ax.set_ylabel("Cohort Month", labelpad=10)
ax.set_title("Retention Rate (%) — Last 24 Cohorts × 18 Months", pad=10)

# Annotate average 3-month and 6-month retention
m3  = cohort_pct.iloc[:, 3].mean()  if 3  in cohort_pct.columns else 0
m6  = cohort_pct.iloc[:, 6].mean()  if 6  in cohort_pct.columns else 0
m12 = cohort_pct.iloc[:, 12].mean() if 12 in cohort_pct.columns else 0
fig.text(0.15, 0.01,
         f"Avg 3-month retention: {m3:.1f}%   |   "
         f"Avg 6-month retention: {m6:.1f}%   |   "
         f"Avg 12-month retention: {m12:.1f}%",
         ha="left", fontsize=10, color=ACCENT2)

plt.tight_layout(rect=[0,0,1,0.96])
save_fig("08_cohort_retention")


[Fig 8] Cohort retention analysis...
  ✓ Saved: eda_outputs/08_cohort_retention.png


In [32]:
# ────────────────────────────────────────────
# FIG 9: Spend Trend & Seasonality by Segment
# ────────────────────────────────────────────
print("[Fig 9] Spend trend & seasonality...")

# Merge orders with RFM segment
orders_seg = (orders_full
              .merge(rfm[["customer_id","Segment"]], on="customer_id", how="left")
              .assign(order_ym = lambda df: df["order_date"].dt.to_period("M")))

monthly_rev_seg = (orders_seg
                   .groupby(["order_ym","Segment"])["payment_value"]
                   .sum().reset_index())

top_segs = ["Champions","Loyal","Promising","At Risk"]

fig, axes = plt.subplots(2, 2, figsize=(20, 10))
fig.suptitle("MONTHLY REVENUE TREND BY CUSTOMER SEGMENT", fontsize=15,
             fontweight="bold", color=ACCENT)

for ax, seg in zip(axes.flatten(), top_segs):
    df_seg = monthly_rev_seg[monthly_rev_seg["Segment"]==seg].sort_values("order_ym")
    color = seg_colors.get(seg, ACCENT)
    ax.fill_between(range(len(df_seg)), df_seg["payment_value"], alpha=0.25, color=color)
    ax.plot(range(len(df_seg)), df_seg["payment_value"], color=color, linewidth=1.5)

    # Trend line
    if len(df_seg) > 3:
        z = np.polyfit(range(len(df_seg)), df_seg["payment_value"], 1)
        p = np.poly1d(z)
        ax.plot(range(len(df_seg)), p(range(len(df_seg))),
                linestyle="--", color="white", linewidth=1, alpha=0.7)
        slope_dir = "↑" if z[0] > 0 else "↓"
        ax.text(0.02, 0.93, f"Trend: {slope_dir} ${z[0]*12:+,.0f}/yr",
                transform=ax.transAxes, fontsize=8.5, color="white")

    ax.set_title(f"Segment: {seg}", pad=8, color=color)
    ax.set_ylabel("Monthly Revenue ($)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))
    ax.set_xticks([])
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout(rect=[0,0,1,0.95])
save_fig("09_segment_revenue_trend")


[Fig 9] Spend trend & seasonality...
  ✓ Saved: eda_outputs/09_segment_revenue_trend.png


In [33]:
# ══════════════════════════════════════════════
# ████  PRESCRIPTIVE  ████
# ══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  [PRESCRIPTIVE] — What should we do?")
print("=" * 60)

# ────────────────────────────────────────────
# FIG 10: Customer Lifetime Value Matrix + Action Plan
# ────────────────────────────────────────────
print("[Fig 10] CLV action matrix...")

# CLV = avg_order_value × purchase_frequency_per_year × expected_lifespan (years)
# Using simplified CLV proxy
ANALYSIS_YEARS = (orders["order_date"].max() - orders["order_date"].min()).days / 365.25

cust_clv = cust_orders.copy()
cust_clv["orders_per_year"] = cust_clv["n_orders"] / max(ANALYSIS_YEARS, 1)
cust_clv["clv_proxy"]       = cust_clv["avg_order_value"] * cust_clv["orders_per_year"] * 3  # 3-year horizon

cust_clv = cust_clv.merge(rfm[["customer_id","Segment","recency"]], on="customer_id", how="left")
cust_clv = cust_clv.dropna(subset=["clv_proxy","recency"])

seg_clv = (cust_clv.groupby("Segment")
           .agg(avg_clv=("clv_proxy","mean"), count=("customer_id","count"),
                avg_recency=("recency","mean"))
           .reindex(seg_order).dropna())

# Action strategies
actions = {
    "Champions":    ("Reward & Amplify",    "VIP early access, referral bonuses, brand ambassador program",  ACCENT3),
    "Loyal":        ("Upsell & Cross-sell", "Premium product recommendations, loyalty tier upgrade offers",  "#8be9fd"),
    "Promising":    ("Nurture",             "Personalised follow-up campaigns, first-repeat incentive",      ACCENT2),
    "New Customer": ("Onboard",             "Welcome series, first purchase discount, usage education",      "#bd93f9"),
    "At Risk":      ("Win-back",            "Time-limited offers, 'We miss you' campaign, survey",          ACCENT),
    "Lost":         ("Re-engage/Retire",    "One-shot re-engagement; if no response, archive",              "#ff5555"),
    "Others":       ("Activate",            "Segmented introductory campaign, product quiz",                 "#555555"),
}

fig = plt.figure(figsize=(22, 12))
fig.suptitle("PRESCRIPTIVE ACTION MATRIX — CUSTOMER CLV STRATEGY", fontsize=16,
             fontweight="bold", color=ACCENT)

gs = GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)

# Bubble chart: recency vs CLV sized by count
ax_bubble = fig.add_subplot(gs[0, :2])
for i, (seg, row) in enumerate(seg_clv.iterrows()):
    color = seg_colors.get(seg, "#555")
    size  = np.sqrt(row["count"]) * 6
    ax_bubble.scatter(row["avg_recency"], row["avg_clv"],
                      s=size, c=color, alpha=0.85, edgecolors="white", linewidths=0.8,
                      label=f"{seg} (n={int(row['count']):,})", zorder=3)
    ax_bubble.annotate(seg,
                       (row["avg_recency"], row["avg_clv"]),
                       textcoords="offset points", xytext=(8,5),
                       fontsize=8, color=color, fontweight="bold")

ax_bubble.set_xlabel("Avg Recency (days since last order)")
ax_bubble.set_ylabel("Avg Projected 3-year CLV ($)")
ax_bubble.set_title("Segment Bubble Chart: Recency vs CLV (bubble = segment size)", pad=8)
ax_bubble.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax_bubble.grid(alpha=0.3)
ax_bubble.axvline(365, color=TEXT_COLOR, linestyle=":", alpha=0.4, linewidth=1)
ax_bubble.text(370, ax_bubble.get_ylim()[0], "1 year inactive →",
               fontsize=7.5, color=TEXT_COLOR, alpha=0.6)
ax_bubble.legend(fontsize=7.5, markerscale=0.7, bbox_to_anchor=(1.01,1), loc="upper left")

# Revenue at risk from At Risk + Lost
ax_risk = fig.add_subplot(gs[0, 2])
at_risk_rev = seg_clv.loc["At Risk","avg_clv"] * seg_clv.loc["At Risk","count"] \
              if "At Risk" in seg_clv.index else 0
lost_rev    = seg_clv.loc["Lost",   "avg_clv"] * seg_clv.loc["Lost",   "count"] \
              if "Lost"    in seg_clv.index else 0
champ_rev   = seg_clv.loc["Champions","avg_clv"] * seg_clv.loc["Champions","count"] \
              if "Champions" in seg_clv.index else 0
loyal_rev   = seg_clv.loc["Loyal","avg_clv"]    * seg_clv.loc["Loyal","count"] \
              if "Loyal" in seg_clv.index else 0

bar_data  = {"At Risk\n(Recoverable)": at_risk_rev,
             "Lost\n(Difficult)":       lost_rev,
             "Champions\n(Protect)":    champ_rev,
             "Loyal\n(Grow)":           loyal_rev}
bar_cols2 = [ACCENT, "#ff5555", ACCENT3, "#8be9fd"]
bars = ax_risk.bar(bar_data.keys(), bar_data.values(), color=bar_cols2, edgecolor="none", width=0.6)
ax_risk.set_title("Revenue at Stake / Opportunity", pad=8)
ax_risk.set_ylabel("Projected 3-yr CLV ($)")
ax_risk.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x/1e6:.1f}M"))
ax_risk.set_xticklabels(bar_data.keys(), fontsize=8)
ax_risk.grid(axis="y", alpha=0.4)
for bar, val in zip(bars, bar_data.values()):
    ax_risk.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(bar_data.values())*0.01,
                 f"${val/1e6:.2f}M", ha="center", fontsize=8, color=TEXT_COLOR)

# Action table
ax_table = fig.add_subplot(gs[1, :])
ax_table.axis("off")
table_data = []
for seg, (strategy, tactic, color) in actions.items():
    if seg in seg_clv.index:
        cnt = int(seg_clv.loc[seg, "count"])
        clv = seg_clv.loc[seg, "avg_clv"]
        rec = seg_clv.loc[seg, "avg_recency"]
        table_data.append([seg, f"{cnt:,}", f"${clv:,.0f}", f"{rec:.0f} days",
                           strategy, tactic])

col_labels = ["Segment","Customers","Avg 3yr CLV","Avg Recency","Strategy","Recommended Tactics"]
table = ax_table.table(cellText=table_data, colLabels=col_labels,
                        loc="center", cellLoc="left")
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 2.0)

for (row, col), cell in table.get_celld().items():
    cell.set_facecolor("#0d1117" if row % 2 == 0 else "#161b22")
    cell.set_edgecolor(GRID_COLOR)
    cell.set_text_props(color=TEXT_COLOR)
    if row == 0:
        cell.set_facecolor("#0f3460")
        cell.set_text_props(color=ACCENT2, fontweight="bold")
    if col == 0 and row > 0:
        seg_name = table_data[row-1][0]
        cell.set_text_props(color=seg_colors.get(seg_name, TEXT_COLOR), fontweight="bold")

ax_table.set_title("Segment-Level Action Playbook", pad=10, fontsize=12,
                    color=ACCENT, fontweight="bold")

plt.tight_layout(rect=[0,0,1,0.95])
save_fig("10_prescriptive_action_matrix")



  [PRESCRIPTIVE] — What should we do?
[Fig 10] CLV action matrix...
  ✓ Saved: eda_outputs/10_prescriptive_action_matrix.png
